# 蒙特卡洛 ε-贪婪算法（MC ε-Greedy）

> **动机：去掉 Exploring Starts 假设**
>
> 在 MC Exploring Starts 中，为了保证每个状态-动作对 $(s, a)$ 都能被充分访问，我们要求**每条轨迹的起始状态与起始动作均随机均匀采样**（即 Exploring Starts）。  
> 然而，该假设在现实场景中往往难以满足——智能体通常只能从固定的初始状态出发，无法任意指定起点。
>
> **解决方案：引入软策略（Soft Policy）**
>
> **软策略**是指对所有状态 $s$ 和动作 $a$，策略满足：
> $$\pi(a \mid s) > 0, \quad \forall s \in \mathcal{S},\ a \in \mathcal{A}$$
> 即任意状态下每个动作都有**非零的被选概率**，从而保证每个已访问状态下的所有动作都能被充分探索。
>
> 在本实现中，软策略与随机起始状态**共同协作**完成探索覆盖：
> - **随机起始状态**：保证所有状态均有机会被访问（覆盖状态空间）；
> - **软策略**：保证在每个访问到的状态下，所有动作都有非零概率被选择（覆盖动作空间）。
>
> 两者结合，确保所有状态-动作对 $(s, a)$ 都能被充分采样，从而无需 Exploring Starts 中"强制以随机动作出发"的额外约束。
>
> 本节采用最常见的软策略实现——**ε-贪婪策略（ε-Greedy Policy）**：
> $$\pi(a \mid s) = \begin{cases} 1 - \varepsilon + \dfrac{\varepsilon}{|\mathcal{A}|}, & a = \arg\max_{a'} Q(s, a') \\[6pt] \dfrac{\varepsilon}{|\mathcal{A}|}, & \text{其他动作} \end{cases}$$
> 其中 $\varepsilon \in (0, 1]$ 控制探索程度：$\varepsilon$ 越大，策略越随机；$\varepsilon \to 0$ 时退化为纯贪婪策略。通过在训练过程中**逐步衰减 $\varepsilon$**，可以在充分探索与策略收敛之间取得平衡。

## 一、导入依赖库

In [ ]:
import numpy as np       # 导入 NumPy 库，用于数值计算和矩阵运算，版本要求 >=1.18
import random            # 导入 Python 标准库 random，用于随机采样
import importlib.util    # 导入 importlib.util，用于按文件路径动态加载模块

# 文件名 "02.1.ModelFree_Env_GridWorldV2.py" 以数字开头且含点号，不符合 Python 标识符规则，无法直接 import
# spec_from_file_location：根据给定模块别名和 .py 文件路径创建模块规格，返回 ModuleSpec 对象
_spec = importlib.util.spec_from_file_location("GridWorld_v2", "02.1.ModelFree_Env_GridWorldV2.py")
# module_from_spec：根据模块规格创建模块对象，此时模块代码尚未执行，返回 module 对象
GridWorld_v2 = importlib.util.module_from_spec(_spec)
# exec_module：执行模块代码完成初始化，之后可通过 GridWorld_v2.GridWorld_v2(...) 正常使用类
_spec.loader.exec_module(GridWorld_v2)
from IPython.display import clear_output  # 导入 IPython 清屏函数，用于在 Jupyter 中刷新输出，避免打印内容过多堆积

## 二、初始化网格世界与策略

In [2]:
gamma = 0.95  # 折扣因子 γ，float，控制未来奖励的衰减比例，本实验设为 0.95

rows = 5      # 网格世界的行数，int，需与 desc 描述字符串的行数一致
columns = 5   # 网格世界的列数，int，需与 desc 描述字符串每行字符数一致

# 使用描述字符串初始化网格世界：'.' 表示普通格，'#' 表示禁止区（得分 -10），'T' 表示目标格（得分 1）
gridworld = GridWorld_v2.GridWorld_v2(
    forbiddenAreaScore=-10,  # 禁止区域的即时奖励，float，负值表示惩罚
    score=1,                 # 目标区域的即时奖励，float
    desc=[".....", ".##..", "..#..", ".#T#.", ".#..."]  # 网格布局描述，list[str]，共 5 行 5 列
)
gridworld.show()  # 以 emoji 打印网格世界布局，无返回值

value = np.zeros(rows * columns)        # 状态价值函数 V(s) 初始化，np.ndarray，shape=(25,)，全零（备用）
qtable = np.zeros((rows * columns, 5))  # 动作价值函数 Q(s,a) 初始化，np.ndarray，shape=(25, 5)，全零

# 随机初始化确定性策略，利用 NumPy 花式索引（Fancy Indexing）一步完成整数索引→one-hot 转换：
# np.random.randint(0,5,size=(rows*columns))：生成 25 个随机动作索引，shape=(25,)，每个值∈[0,4]，int
# np.eye(5)[整数数组]：花式索引——对数组中每个整数 i，取 np.eye(5) 第 i 行（动作 i 的 one-hot 向量）
#   并沿第 0 轴堆叠，等价于 np.stack([np.eye(5)[i] for i in idx])，但由 C 实现、效率更高
# 结果 policy：np.ndarray，shape=(25, 5)，每行恰有一个 1（对应该状态随机选定的动作），其余为 0
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]
gridworld.showPolicy(policy)  # 可视化初始随机策略，无返回值

⬜️⬜️⬜️⬜️⬜️
⬜️🚫🚫⬜️⬜️
⬜️⬜️🚫⬜️⬜️
⬜️🚫✅🚫⬜️
⬜️🚫⬜️⬜️⬜️
➡️🔄⬅️➡️⬆️
⬅️⏫️🔄⬇️🔄
⬅️⬇️⏫️➡️⬇️
➡️⏪✅⏫️⬆️
⬅️🔄⬇️⬅️🔄


## 三、ε-贪婪 MC 策略迭代

In [ ]:
policy = np.eye(5)[np.random.randint(0, 5, size=(rows * columns))]  # 重新随机初始化策略，shape=(25, 5)，one-hot 编码
gridworld.show()              # 打印网格世界布局
gridworld.showPolicy(policy)  # 打印初始随机策略
print("random policy")        # 提示当前使用随机初始化策略

# 每条轨迹的采样步数，int
# 注意：本实现中每个回合（episode）并不在到达终止状态时自然结束，
# 而是固定采样 trajectorySteps 步作为一批经验数据，不提前截断
# 总采样步数 = num_episodes × trajectorySteps = 200 × 20000 = 4,000,000 步
#
# 为什么步数要远大于格子数（25）？
# MC 方法用"多次采样的均值"估计 Q(s,a)，单次采样噪声极大（高方差），
# 必须对每个状态-动作对积累足够多的访问次数才能得到可靠估计：
#   - Q 表共有 25×5=125 个状态-动作对需要估计
#   - 访问分布不均匀：靠近目标的状态被频繁访问，角落/障碍附近的状态访问极少，
#     冷门对需要更多总步数才能积累到足够样本
#   - 本实现为非重置式连续轨迹（到达目标后不重置，继续行走），
#     20000 步约等于在棋盘上绕行 ~160 圈（20000÷125），
#     才能让各状态-动作对的 Q 值均值收敛到可靠估计
trajectorySteps = 20000
epsilon = 0.1            # ε-greedy 的初始探索率，float，控制随机探索比例，越大探索越多
qtable = np.zeros((rows * columns, 5))  # 初始化 Q 表，np.ndarray，shape=(25, 5)，全零
# 训练的总回合数，int；每回合采样一条固定长度轨迹，然后做一次 Q 表更新与策略改进
num_episodes = 200

for episode in range(num_episodes):  # 按回合循环训练，episode 为当前回合编号，int，范围 [0, 200)
    # ε 衰减策略——体现探索与利用的权衡（Exploration-Exploitation Tradeoff）：
    #   训练初期：ε 大 → 大量随机探索，Q 表从零开始积累经验，避免过早固化在局部最优动作上
    #   训练后期：ε 小 → 主要利用已学到的 Q 值，策略趋近贪婪并稳定收敛
    #   始终保留 ε_min=0.001：维持软策略性质（所有动作概率 > 0），
    #   防止某些状态-动作对因概率为 0 而永远停止更新，保留持续修正错误的能力
    if epsilon > 0.001:  # 探索率线性衰减：每回合减少 0.001
        epsilon -= 0.001  # 减小探索率，float，策略逐渐向贪婪方向收敛
    else:
        epsilon = 0.001   # 探索率下限，float，始终保留少量探索以避免策略退化

    p1 = 1 - epsilon * (4 / 5)  # 最优动作的选择概率，float：p1 = 1 - ε*(|A|-1)/|A|，|A|=5
    p0 = epsilon / 5             # 非最优动作的选择概率，float：p0 = ε/|A|，均匀分配给各非最优动作

    print(f"{'=' * 55}")  # 打印分隔线，标识新一回合的开始
    print(f"  第 {episode + 1:>3d} / {num_episodes} 回合  |  ε={epsilon:.4f}  p1={p1:.4f}  p0={p0:.4f}")
    print(f"{'=' * 55}")  # 打印分隔线下边框
    print(f"  轨迹步数：{trajectorySteps}")

    # 构建"值→概率"映射字典，dict{int: float}：
    #   one-hot policy 中值为 1 的位置代表最优动作 → 映射为 p1（较大概率）
    #   值为 0 的位置代表非最优动作              → 映射为 p0（较小概率）
    d = {1: p1, 0: p0}

    # 将确定性 one-hot policy 转换为 ε-greedy 概率策略，分三步理解：
    #
    # 第一步：d.get 是字典的查询方法，d.get(key) 等价于 d[key]
    #   例：d.get(1) → p1，d.get(0) → p0
    #
    # 第二步：np.vectorize(d.get) 将 d.get 包装成"逐元素函数"
    #   原本 d.get 只能处理单个标量，np.vectorize 让它能对 ndarray 的每个元素依次调用
    #   返回一个可调用对象 vfunc，等价于：lambda arr: np.array([[d.get(x) for x in row] for row in arr])
    #
    # 第三步：vfunc(policy) 对 policy 中每个元素执行 d.get
    #   policy shape=(25, 5)，元素为 0 或 1（one-hot）
    #   → 每个 1（最优动作）被替换为 p1，每个 0（非最优动作）被替换为 p0
    #
    # 示例（假设某状态最优动作为动作 2，|A|=5，ε=0.1，p1≈0.92，p0=0.02）：
    #   policy[s]         = [0,    0,    1,    0,    0   ]
    #   policy_epsilon[s] = [0.02, 0.02, 0.92, 0.02, 0.02]  → 行和 = 1
    #
    # policy_epsilon：np.ndarray，shape=(25, 5)，dtype=float，每行为合法概率分布（行和=1）
    #   第 0 维：25 个状态；第 1 维：5 个动作对应的 ε-greedy 选择概率
    policy_epsilon = np.vectorize(d.get)(policy)

    i = random.randint(0, 24)  # 随机选择起始状态，int，范围 [0, 24]，实现探索性出发（Exploring Starts）
    j = random.randint(0, 4)   # 随机选择起始动作，int，范围 [0, 4]，确保所有状态-动作对都有机会被访问

    cnt = [0 for _ in range(25)]  # 各状态的访问次数计数器，list[int]，长度 25，全零，用于调试
    qtable_rewards = [[0 for _ in range(5)] for _ in range(rows * columns)]  # 各状态-动作对的累计折扣回报，list[list[float]]，shape=(25, 5)
    qtable_nums    = [[0 for _ in range(5)] for _ in range(rows * columns)]  # 各状态-动作对的访问次数，list[list[int]]，shape=(25, 5)

    # 从状态 i、动作 j 出发，按 ε-greedy 策略 policy_epsilon 采样 trajectorySteps 步轨迹
    # 返回值：list[tuple]，长度为 trajectorySteps+1，每个元组为 (nowState, nowAction, score, nextState, nextAction)
    Trajectory = gridworld.getTrajectoryScore(
        nowState=i, action=j, policy=policy_epsilon, steps=trajectorySteps
    )
    clear_output(wait=True)  # 清除 Jupyter 输出区域，避免每回合打印内容叠加

    score = 0  # 折扣累计回报 G 的初始值，float，从轨迹末端向前递推时的起点

    # 从轨迹末端（第 trajectorySteps 步）向前逆向遍历，计算每步的折扣累计回报
    for k in range(trajectorySteps, -1, -1):
        tmpstate, tmpaction, tmpscore, _, _ = Trajectory[k]  # 解包第 k 步的状态（int）、动作（int）、即时奖励（float）
        cnt[tmpstate] += 1                # 该状态访问次数加 1，int
        score = score * gamma + tmpscore  # 递推折扣累计回报 G_k = r_k + γ*G_{k+1}，float
        qtable_rewards[tmpstate][tmpaction] += score  # 累加该状态-动作对的折扣回报，float
        qtable_nums[tmpstate][tmpaction] += 1         # 该状态-动作对的访问次数加 1，int
        # Every-Visit MC：用样本均值更新 Q 值，Q(s,a) ← 累计回报之和 / 访问次数，float
        qtable[tmpstate][tmpaction] = (
            qtable_rewards[tmpstate][tmpaction] / qtable_nums[tmpstate][tmpaction]
        )

    values = []  # 存储每个状态在 ε-greedy 策略下的状态价值，list[float]，长度 25
    for i in range(25):   # 遍历所有状态，i 为状态编号，int
        v = 0             # 当前状态价值初始化，float
        for j in range(5):  # 遍历所有动作，j 为动作编号，int
            v += policy_epsilon[i][j] * qtable[i][j]  # V(s) = Σ_a π(a|s)*Q(s,a)，float
        values.append(v)  # 将当前状态价值追加到列表

    # 打印状态价值矩阵，np.ndarray，shape=(5, 5)，直观展示当前策略下各格子的价值分布
    print(np.array(values).reshape(5, 5))

    print(f"  当前策略：")  # 打印策略可视化标签
    gridworld.showPolicy(policy)         # 可视化当前确定性贪婪策略（转换前的 policy）
    print(f"  状态价值均值：{np.array(values).mean():.6f}")  # 打印所有状态价值的均值，float，监控整体学习进度

    # 策略改进：根据当前 Q 表贪婪地选择每个状态的最优动作，构造新的 one-hot 确定性策略
    policy = np.eye(5)[np.argmax(qtable, axis=1)]  # shape=(25, 5)，one-hot 编码
    # 将更新后的贪婪策略转换为 ε-greedy 概率策略，供下一回合采样使用
    policy_epsilon = np.vectorize(d.get)(policy)   # shape=(25, 5)，float，每行和为 1

print(f"\n{'*' * 55}")  # 打印最终分隔线
print(f"  ε-贪婪 MC 训练完成，共训练 {num_episodes} 回合")  # 打印训练总回合数
print(f"  最终 ε={epsilon:.4f}  |  最终状态价值均值：{np.array(values).mean():.6f}")  # 打印最终关键指标
print(f"{'*' * 55}")

[[-1.15546883e-02 -2.60055597e-03 -4.90606254e-04  7.32429639e+00
   7.70005801e+00]
 [ 8.10985860e+00 -1.46188646e+00  8.98894888e+00  1.23883261e-02
   8.11580545e+00]
 [ 8.54246804e+00  8.99739952e+00  1.99882296e+01  8.99621580e+00
   8.54467363e+00]
 [ 8.99368516e+00  1.99906144e+01  1.98269725e+01  1.95968941e+01
   8.99161071e+00]
 [ 8.04625061e+00  1.89813036e+01  1.99875246e+01  8.91622173e+00
   8.54150071e+00]]
⬇️➡️➡️➡️⬇️
⬇️⏩️⏬⬇️⬇️
➡️⬇️⏬⬅️⬅️
➡️⏩️✅⏪⬅️
➡️⏩️⬆️⬆️⬆️
9.428655519968848


In [ ]:
gridworld.showPolicy(policy)  # 可视化最终收敛后的策略，展示每个状态下的最优动作方向

⬇️➡️➡️➡️⬇️
⬇️⏩️⏬⬇️⬇️
➡️⬇️⏬⬅️⬅️
➡️⏩️✅⏪⬅️
➡️⏩️⬆️⬆️⬆️
